## Agent

In [ ]:
import os
from dotenv import load_dotenv
from datetime import date, timedelta
import random
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from pydantic import BaseModel
from langchain.messages import HumanMessage
from langchain.tools import tool
import sqlite3
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
import requests

from rich import print
import sys
sys.path.append('..')
from utils.helper import pretty_print_messages

load_dotenv()

True

In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not os.getenv("TAVILY"):
    raise ValueError("TAVILY environment variable is not set.")
TAVILY = os.getenv("TAVILY")

llm = ChatOpenAI(model="gpt-5-nano")

In [3]:
connection = sqlite3.connect("data/sales.db")

cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS sales (
        order_id INTEGER PRIMARY KEY,
        company TEXT,
        category TEXT,
        product_name TEXT,
        country TEXT,
        order_date Date,
        cost_price REAL,
        selling_price REAL,
        quantity INTEGER,
        currency TEXT
    )
""")

In [4]:
web_search = TavilySearch(max_results=1, tavily_api_key=TAVILY)

In [5]:
@tool
def insert_sales_record_in_database(order_id: int, company: str, category: str, product_name: str, country: str, order_date: str, cost_price: float, selling_price: float, quantity: int, currency: str) -> str:
    """
    Insert a new sales record into the sales database.
    Use this tool when the user wants to add sales data.
    """
    connection = sqlite3.connect("data/sales.db")
    cursor = connection.cursor()
    cursor.execute("""
        INSERT INTO sales (order_id, company, category, product_name, country, order_date, cost_price, selling_price, quantity, currency)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (order_id, company, category, product_name, country, order_date, cost_price, selling_price, quantity, currency))
    connection.commit()
    connection.close()
    return "Sales record inserted successfully."

In [6]:
from datetime import date, timedelta
import random

companies = [
    "Amazon",
    "Flipkart",
    "Myntra",
    "Nykaa",
    "Croma",
    "Noon",
    "EMart",
    "Walmart",
    "Snapdeal",
    ]

products = {
    "Electronics": [
        ("Apple iPhone 15", 55000, 70000),
        ("Samsung Galaxy S24", 45000, 60000),
        ("Sony Headphones", 5000, 8000),
        ("Dell Laptop", 45000, 60000),
        ("Boat Smart Watch", 1500, 3000),
    ],

    "Fashion": [
        ("Nike Running Shoes", 2500, 5000),
        ("Levis Jeans", 1200, 2500),
        ("Allen Solly Shirt", 800, 1800),
        ("Puma T-Shirt", 600, 1500),
        ("Adidas Jacket", 3000, 6000),
    ],

    "Beauty": [
        ("Lakme Lipstick", 300, 600),
        ("Maybelline Mascara", 250, 550),
        ("Loreal Shampoo", 300, 700),
        ("The Ordinary Serum", 500, 1200),
        ("Mamaearth Face Wash", 200, 450),
    ],

    "Home Appliances": [
        ("Philips Mixer Grinder", 2500, 4500),
        ("LG Microwave Oven", 8000, 12000),
        ("Samsung Refrigerator", 25000, 40000),
        ("Prestige Cooker", 1000, 2000),
        ("Havells Fan", 1500, 3000),
    ]
}


countries = ["India", "UAE", "USA", "UK", "Singapore"]


def generate_sales_orders(number_of_records=100):
    sales_orders = []
    start_date = date(2023, 1, 1)

    for order_id in range(1, number_of_records + 1):
        company = random.choice(companies)
        category = random.choice(list(products.keys()))
        product_name, cost_range, selling_range = random.choice(products[category])
        quantity = random.randint(1, 10)
        cost_price = round(random.uniform(cost_range * 0.8, cost_range * 1.1), 2)
        selling_price = round(random.uniform(selling_range * 0.9, selling_range * 1.2), 2)
        order_date = start_date + timedelta(days=random.randint(0, 1000))
        country = random.choice(countries)
        sales_orders.append((order_id, company, category, product_name, country, order_date.isoformat(), cost_price, selling_price, quantity, 'USD'))
    return sales_orders


# Generate data
sales_data = generate_sales_orders(200)

# Print first 5 records
for row in sales_data[:5]:
    print(row)

(1, 'Amazon', 'Beauty', 'Lakme Lipstick', 'UK', '2024-04-18', 320.19, 673.01, 9, 'USD')

(2, 'Croma', 'Fashion', 'Nike Running Shoes', 'USA', '2024-06-03', 2277.31, 5674.73, 1, 'USD')

(3, 'Walmart', 'Beauty', 'Mamaearth Face Wash', 'UAE', '2023-02-25', 187.88, 490.85, 8, 'USD')

(4, 'Walmart', 'Beauty', 'Loreal Shampoo', 'India', '2025-04-22', 257.53, 781.86, 7, 'USD')

(5, 'EMart', 'Home Appliances', 'Havells Fan', 'Singapore', '2024-02-15', 1218.55, 2742.51, 1, 'USD')

In [13]:
for order_id, company, category, product_name, country, order_date, cost_price, selling_price, quantity, currency in sales_data:
    result = insert_sales_record_in_database.invoke({
        "order_id": order_id,
        "company": company,
        "category": category,
        "product_name": product_name,
        "country": country,
        "order_date": order_date,
        "cost_price": cost_price,
        "selling_price": selling_price,
        "quantity": quantity,
        "currency": currency
    })


In [7]:
# -----------------------------
# Get database schema
# -----------------------------
def get_database_schema():
    connection = sqlite3.connect("data/sales.db")
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    schema = []

    for (table_name,) in tables:
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        schema.append(f"Table: {table_name}")
        for column in columns:
            schema.append(f"  - {column[1]} ({column[2]})")
    cursor.close()
    return "\n".join(schema)

In [8]:
get_database_schema()

'Table: sales\n  - order_id (INTEGER)\n  - company (TEXT)\n  - category (TEXT)\n  - product_name (TEXT)\n  - country (TEXT)\n  - order_date (Date)\n  - cost_price (REAL)\n  - selling_price (REAL)\n  - quantity (INTEGER)\n  - currency (TEXT)'

In [42]:
# -----------------------------
# SQL generation tool
# -----------------------------
@tool
def generate_sql_query(question: str) -> str:
    """
    Convert a user's natural-language question into a SQL query.

    Use this tool when the user asks a question about data
    stored in the database.
    """
    schema = get_database_schema()

    prompt = f"""
        You are an expert SQL query generator for SQLite3 dialect.
        Convert the user's natural-language question into a SQL query.
        Database schema:
        {schema}

        User question:
        {question}

        Rules:
        1. Generate valid SQLite SQL.
        2. Use only tables and columns present in the schema.
        3. Do not invent tables or columns.
        4. Return ONLY the SQL query.
        5. Do not include markdown code fences.
        6. Only generate SELECT queries.

    SQL:
    """

    response = llm.invoke(prompt)

    return response.content.strip()

In [43]:
query = generate_sql_query.invoke("What was the total number of sales per country in 2023?")

In [44]:
print(query)

SELECT country,
       COUNT(*) AS total_sales
FROM sales
WHERE strftime('%Y', order_date) = '2023'
GROUP BY country
ORDER BY total_sales DESC;

In [45]:
@tool
def run_sales_query(query: str) -> str:
    """
    Use this tool to execute a read-only SQL query against the sales database.
    This tool is intended for analyzing sales data, including questions about sales,
    revenue, products, trends, over time, and other related metrics.
    Args:
        query (str): A read-only SQL query in SQLite3 dialect.
    """
    connection = sqlite3.connect("data/sales.db") 
    cursor = connection.cursor()
    # Basic safety for the demo
    forbidden = ["insert", "update", "delete", "drop", "alter", "truncate" ]

    normalized = query.lower()

    if any(word in normalized for word in forbidden):
        return "Only read-only SQL queries are allowed."

    try:
        result = cursor.execute(query)
        rows = result.fetchall()
        connection.close()
        return rows
    except Exception as e:
        return f"SQL error: {e}"

In [46]:
res = run_sales_query.invoke(query)

In [47]:
res

[('Singapore', 20), ('UK', 15), ('UAE', 15), ('India', 13), ('USA', 9)]

## Add currency Converter Tool

In [48]:
@tool
def currency_converter(amount: float, base_currency: str, quote_currency: str) -> float:
    """
    Convert an amount from one currency to another used when the user asks about sales in different currencies.
    Or want to convert the analysis in specific currencies.

    Args:
        amount: Amount to convert.
        base_currency: Three-letter source currency code, e.g. USD.
        quote_currency: Three-letter target currency code, e.g. SGD.
    Returns:
        Converted amount.
    """

    url = "https://api.frankfurter.dev/v2/rates"

    params = {
        "base": base_currency.upper(),
        "quotes": quote_currency.upper()
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    rate = data[0]["rate"]
    converted_amount = amount * rate

    return round(converted_amount, 2)

In [49]:
converted_amount = currency_converter.invoke(input={"amount": 10, "base_currency": "USD", "quote_currency": "INR"})
print(converted_amount)

952.1

In [50]:
class UserContext(BaseModel):
    user_id: str
    role: str

In [51]:
agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[
        generate_sql_query,
        run_sales_query,
        web_search,
        currency_converter
    ],
    system_prompt="""You are an enterprise Business Analyst Agent.
Please utilize all available tools to assist the user in analyzing internal sales data and market trends.
Don't overcomplicate things by asking follow up questions, just provide the analysis what you feel is right.""",
    context_schema=UserContext,
    checkpointer=InMemorySaver()
)

In [52]:
context = UserContext(user_id="user_123", role="bussiness_analyst")
bussiness_analyst = {"configurable": {"thread_id": "user_12345"}}

In [53]:
result = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the revenue by country per product category?")]},
        config=bussiness_analyst,
        context=context
)

In [54]:
print(result["messages"][-1].content)

Here is the revenue by country broken down by product category (in your reporting currency, likely USD), plus 
totals.

India
- Electronics: 3,003,628.65
- Fashion: 138,319.85
- Beauty: 32,200.64
- Home Appliances: 1,433,838.91
- Total: 4,607,988.05

Singapore
- Electronics: 1,639,871.63
- Fashion: 232,864.32
- Beauty: 39,750.65
- Home Appliances: 576,270.50
- Total: 2,488,757.10

UAE
- Electronics: 1,043,251.21
- Fashion: 223,805.27
- Beauty: 22,927.86
- Home Appliances: 868,851.14
- Total: 2,158,835.48

United Kingdom
- Electronics: 1,763,089.65
- Fashion: 202,080.51
- Beauty: 36,658.28
- Home Appliances: 947,513.01
- Total: 2,949,341.45

United States
- Electronics: 103,487.72
- Fashion: 207,024.81
- Beauty: 59,598.01
- Home Appliances: 706,147.08
- Total: 1,076,257.62

Key insights
- Electronics is the largest revenue driver in most countries (India, Singapore, UAE, UK). In the USA, Home 
Appliances is the leading category (706k) followed by Fashion (207k) and Electronics (103k).
- By total revenue, India leads (≈4.61M), followed by the UK (≈2.95M), Singapore (≈2.49M), UAE (≈2.16M), and the 
USA (≈1.08M).

If you’d like, I can generate a CSV pivot or a chart (e.g., stacked bars by country) for quick visualization.

In [55]:
result_3 = agent.invoke(
    {"messages": [
        HumanMessage(content="I would like to analyze year over year revenue, profit/loss trends in percentages also")]},
        config=bussiness_analyst,
        context=context
)

In [56]:
print(result_3["messages"][-1].content)

Here are the year-over-year (YoY) trends for revenue and profitability (gross profit and gross margin). I’m using 
gross profit as the profitability proxy since your data doesn’t include operating expenses, taxes, etc.

Year 2023
- Revenue: 4,828,353.07
- Gross Profit: 1,963,017.45
- Gross Margin: 40.66%
- YoY revenue: N/A (base year)

Year 2024
- Revenue: 3,878,038.26
- Gross Profit: 1,520,274.45
- Gross Margin: 39.20%
- YoY revenue change vs 2023: -19.68%
- YoY gross margin change: -3.58 percentage points

Year 2025
- Revenue: 4,576,788.37
- Gross Profit: 1,796,271.51
- Gross Margin: 39.25%
- YoY revenue change vs 2024: +18.02%
- YoY gross margin change: +0.12 percentage points

Key insights
- 2024 saw a sharp revenue decline compared with 2023 (-19.7%), accompanied by a margin squeeze (gross margin down 
~3.6 pp).
- 2025 revenue rebounded strongly (+18.0% vs 2024), with gross margin stabilizing and showing a small uptick (~0.12
pp). Gross margin remains around 39–40%.
- If net profitability is a criterion, you’ll want to incorporate operating expenses, taxes, and any other costs to
compute net profit and net margin. I can compute a net-profit YoY view if you provide that data.

Next steps (optional)
- Provide a CSV or dashboard-ready export for these yearly figures.
- Break down YoY trends by country or product category to identify drivers behind the 2024 dip and 2025 rebound.
- Add a moving-average or CAGR view for longer-term trend analysis (e.g., 3-year CAGR).
- If you want, I can compute net profit margin and net profit YoY once you share operating expenses.

In [57]:
pretty_print_messages(result_3)

Message 1
Role : Human
--------------------------------------------------------------------------------
What is the revenue by country per product category?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : generate_sql_query
    Args : {'question': 'What is the revenue by country per product category?'}
    ID   : call_ljJr7sozJ8INJtDjlBq9OGgS
Message 3
Role : Tool
--------------------------------------------------------------------------------
SELECT country, category, SUM(quantity * selling_price) AS revenue
FROM sales
GROUP BY country, category
ORDER BY country, category;

Tool Name : generate_sql_query
Tool Call : call_ljJr7sozJ8INJtDjlBq9OGgS
Message 4
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : run_sales_query
    Args : {'query': 'SELECT country, category, SUM(quantity * selling_price) AS revenue\nFROM sales\nGROUP BY country, categ

In [58]:
result_4 = agent.invoke(
    {"messages": [
        HumanMessage(content="Can you provide this analysis in INR instead of USD?")]},
        config=bussiness_analyst,
        context=context
)

In [59]:
print(result_4["messages"][-1].content)

Converted to INR (approximate, using a blended rate around 95.2 INR per 1 USD based on the dataset).

Revenue by country per product category (INR)

India
- Electronics: 285,975,483.77
- Fashion: 13,169,432.92
- Beauty: 3,065,822.93
- Home Appliances: 136,515,802.62
- Total: 438,726,542.24

Singapore
- Electronics: 156,132,177.89
- Fashion: 22,171,011.91
- Beauty: 3,784,659.39
- Home Appliances: 54,866,714.30
- Total: 236,954,563.49

UAE
- Electronics: 99,327,947.70
- Fashion: 21,308,499.76
- Beauty: 2,182,961.55
- Home Appliances: 82,723,317.04
- Total: 205,542,726.05

United Kingdom
- Electronics: 167,863,765.58
- Fashion: 19,240,085.36
- Beauty: 3,490,234.84
- Home Appliances: 90,212,713.68
- Total: 280,806,799.46

United States
- Electronics: 9,853,065.82
- Fashion: 19,710,832.16
- Beauty: 5,674,326.53
- Home Appliances: 67,232,263.49
- Total: 102,470,487.99

Year-over-year revenue and profitability in INR

Revenue (INR)
- 2023: approximately 459,707,495.79
- 2024: approximately 369,228,022.73
- 2025: approximately 435,756,020.71
- YoY revenue changes:
  - 2024 vs 2023: about -19.68%
  - 2025 vs 2024: about +18.01%

Profitability (gross) in INR
- Gross Profit (INR):
  - 2023: approximately 186,898,891.42
  - 2024: approximately 144,745,330.38
  - 2025: approximately 171,022,990.47
- Gross Margin (as % of revenue, INR basis):
  - 2023: 40.66%
  - 2024: 39.20%
  - 2025: 39.25%
- YoY gross profit changes:
  - 2024 vs 2023: about -22.57%
  - 2025 vs 2024: about +18.16%

Notes and caveats
- FX rate: Values are approximate. I used a blended rate around 95.2 INR per USD based on the provided conversions.
If you have a specific date-based FX rate or currency (e.g., use end-of-quarter rate), I can recompute precisely.
- These figures use gross profit as profitability (no operating expenses, taxes, or interest included). If you 
share operating costs, I can calculate net profit and net margin in INR.
- If you’d like, I can export these figures to CSV, generate charts (e.g., stacked bars by country or line charts 
for YoY trends), or break down YoY trends by country or by category for deeper insights.

In [60]:
pretty_print_messages(result_4)

Message 1
Role : Human
--------------------------------------------------------------------------------
What is the revenue by country per product category?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : generate_sql_query
    Args : {'question': 'What is the revenue by country per product category?'}
    ID   : call_ljJr7sozJ8INJtDjlBq9OGgS
Message 3
Role : Tool
--------------------------------------------------------------------------------
SELECT country, category, SUM(quantity * selling_price) AS revenue
FROM sales
GROUP BY country, category
ORDER BY country, category;

Tool Name : generate_sql_query
Tool Call : call_ljJr7sozJ8INJtDjlBq9OGgS
Message 4
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : run_sales_query
    Args : {'query': 'SELECT country, category, SUM(quantity * selling_price) AS revenue\nFROM sales\nGROUP BY country, categ

In [23]:
context = UserContext(user_id="user_678", role="market_analyst")
market_analyst = {"configurable": {"thread_id": "user_678"}}

In [ ]:
result_5 = agent.invoke(
    {"messages": [
        HumanMessage(content="""Tell in brief summary, dont ask follow-up question, return what you feel is the best answer to the user query.:
        Based on external data sources, which companies are performing best globally in below metrics for 2026 YTD
        Give top company for each metric separately:
        - YoY revenue growth
        - Net profit margin
        """)]},
        config=market_analyst,
        context=context
)

In [ ]:
print(result_5["messages"][-1].content)

Here are the top performers from external sources for 2026 year-to-date, one per metric:

- YoY revenue growth: Eli Lilly — about 56% YoY revenue growth in Q1 2026. Source: Time (America’s Best Companies 
of 2026), July 2026.

- Net profit margin: Alphabet — net profit margin around 38% (TTM) as of mid-2026. Source: Datarails (Most 
Profitable Companies in the World: Top 10 in 2026).

In [ ]:
pretty_print_messages(result_5)

Message 1
Role : Human
--------------------------------------------------------------------------------
Tell in brief summary, dont ask follow-up question, return what you feel is the best answer to the user query.:
        Based on external data sources, which companies are performing best globally in below metrics for 2026 YTD
        Give top company for each metric separately:
        - YoY revenue growth
        - Net profit margin
        
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : tavily_search
    Args : {'query': '2026 year-to-date YoY revenue growth top companies', 'time_range': 'year', 'search_depth': 'advanced'}
    ID   : call_lcIJ8tZQ5OmyZNhoCOMVZC7D
  • Name : tavily_search
    Args : {'query': '2026 net profit margin top companies YTD', 'time_range': 'year', 'search_depth': 'advanced'}
    ID   : call_btsJ5hVDg531NVaRVMYixXBx
Message 3
Role : Tool
-----------------------------------------

In [61]:
result_6 = agent.invoke(
    {"messages": [
        HumanMessage(content="I would like to further drill the analysis down by country")]},
        config=bussiness_analyst,
        context=context
)

In [62]:
print(result_6["messages"][-1].content)

Here’s a country-by-country drill-down, with figures converted to INR (using the same USD-to-INR rate used earlier:
95.2 INR per 1 USD). Metrics shown per year: Revenue INR, Gross Profit INR, Gross Margin %, plus YoY changes in 
revenue and gross margin.

India
- 2023: Revenue INR 144,717,919.19; Gross Profit INR 58,235,225.16; Gross Margin 40.24%
- 2024: Revenue INR 107,066,739.98; Gross Profit INR 46,367,363.73; Gross Margin 43.31%
  - YoY Revenue: -26.02%; YoY Gross Margin: +7.62 pp
  - Revenue Delta INR: -37,651,179.22
- 2025: Revenue INR 186,895,803.19; Gross Profit INR 74,779,099.25; Gross Margin 40.01%
  - YoY Revenue: +74.56%; YoY Gross Margin: -7.61 pp
  - Revenue Delta INR: +79,829,063.22

Singapore
- 2023: Revenue INR 87,080,193.98; Gross Profit INR 36,731,139.76; Gross Margin 42.18%
- 2024: Revenue INR 99,039,375.06; Gross Profit INR 33,208,588.39; Gross Margin 33.53%
  - YoY Revenue: +13.73%; YoY Gross Margin: -20.51 pp
  - Revenue Delta INR: +11,959,181.08
- 2025: Revenue INR 50,810,106.87; Gross Profit INR 15,711,431.96; Gross Margin 30.92%
  - YoY Revenue: -48.70%; YoY Gross Margin: -7.78 pp
  - Revenue Delta INR: -48,292,268.19

United Arab Emirates (UAE)
- 2023: Revenue INR 81,780,847.90; Gross Profit INR 34,810,920.54; Gross Margin 42.57%
- 2024: Revenue INR 48,750,767.13; Gross Profit INR 19,039,241.26; Gross Margin 39.05%
  - YoY Revenue: -40.39%; YoY Gross Margin: -8.25 pp
  - Revenue Delta INR: -33,030,080.78
- 2025: Revenue INR 74,989,522.66; Gross Profit INR 32,433,564.24; Gross Margin 43.25%
  - YoY Revenue: +53.82%; YoY Gross Margin: +4.77 pp
  - Revenue Delta INR: +26,238,755.54

United Kingdom (UK)
- 2023: Revenue INR 113,605,313.02; Gross Profit INR 42,406,273.16; Gross Margin 37.33%
- 2024: Revenue INR 61,741,075.59; Gross Profit INR 21,680,031.18; Gross Margin 35.11%
  - YoY Revenue: -45.65%; YoY Gross Margin: -5.93 pp
  - Revenue Delta INR: -51,864,237.43
- 2025: Revenue INR 105,621,317.42; Gross Profit INR 39,923,090.09; Gross Margin 37.80%
  - YoY Revenue: +71.07%; YoY Gross Margin: +7.64 pp
  - Revenue Delta INR: +43,880,241.83

United States (USA)
- 2023: Revenue INR 32,474,938.16; Gross Profit INR 14,694,702.62; Gross Margin 45.25%
- 2024: Revenue INR 52,591,284.59; Gross Profit INR 24,434,903.08; Gross Margin 46.46%
  - YoY Revenue: +61.94%; YoY Gross Margin: +2.68 pp
  - Revenue Delta INR: +20,116,346.43
- 2025: Revenue INR 17,393,502.67; Gross Profit INR 8,157,862.22; Gross Margin 46.90%
  - YoY Revenue: -66.93%; YoY Gross Margin: +0.95 pp
  - Revenue Delta INR: -35,197,781.92

Key takeaways by country
- India shows a strong rebound in 2025 after a dip in 2024, with margin around 40% each year.
- Singapore and UK experience large 2024 dips with partial recoveries in 2025; margins fluctuating notably, 
indicating potential mix shifts or pricing changes.
- UAE stabilized in 2025 with a higher margin, after a soft 2024.
- USA saw a substantial drop in 2025 despite solid performance in 2024; margins remain healthy but revenue 
volatility is high.

Next steps (optional)
- Create country-specific dashboards or CSV exports to enable visual inspection (line charts for revenue and gross 
margin, stacked bars for revenue by product category within each country).
- Drill down by product category within each country to identify drivers of growth or decline.
- If you want net profitability, provide operating expenses so I can compute country-level net profit and net 
margin per year.
- I can also generate CAGR, moving averages, or YoY tables tailored to each country.

Would you like me to export this drill-down to CSV and/or generate per-country charts?

In [63]:
pretty_print_messages(result_6)

Message 1
Role : Human
--------------------------------------------------------------------------------
What is the revenue by country per product category?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : generate_sql_query
    Args : {'question': 'What is the revenue by country per product category?'}
    ID   : call_ljJr7sozJ8INJtDjlBq9OGgS
Message 3
Role : Tool
--------------------------------------------------------------------------------
SELECT country, category, SUM(quantity * selling_price) AS revenue
FROM sales
GROUP BY country, category
ORDER BY country, category;

Tool Name : generate_sql_query
Tool Call : call_ljJr7sozJ8INJtDjlBq9OGgS
Message 4
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : run_sales_query
    Args : {'query': 'SELECT country, category, SUM(quantity * selling_price) AS revenue\nFROM sales\nGROUP BY country, categ